# Adquisición y Cruce de Datos
**Proyecto:** Predicción de Satisfacción en Aerolíneas según Servicio y Perfil Sociodemográfico.
**Objetivo de este notebook:** Leer las fuentes originales, realizar el cruce estadístico por edad y género, y consolidar el diccionario de datos.

In [2]:
import pandas as pd
import numpy as np
import matplotlib

# Esto obliga a matplotlib a usar un motor estático y evita que el Kernel explote
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
# Configuración para ver todas las columnas si el dataset es muy ancho
pd.set_option('display.max_columns', None)

Se cargan los datasets desde la carpeta `/data`. 
* **Fuente 1:** Pasajeros de Aerolínea (Kaggle)
* **Fuente 2:** Censo de Ingresos Adultos (Kaggle)

In [4]:
# Leer Fuente 1: Pasajeros
pasajeros = pd.read_csv('../data/airline_passenger_satisfaction.csv')
print(f"Dimensiones de Pasajeros: {pasajeros.shape}")
display(pasajeros.head(3))

# Leer Fuente 2: Censo
censo = pd.read_csv('../data/adult_census_income.csv')
print(f"\nDimensiones de Censo: {censo.shape}")
display(censo.head(3))

Dimensiones de Pasajeros: (103904, 25)


,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,3,1,5,3,5,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,3,3,1,3,1,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,2,2,5,5,5,5,4,3,4,4,4,5,0,0.0,satisfied



Dimensiones de Censo: (32561, 15)


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K


Los nombres de las columnas varían entre los datasets (ej. `Age` vs `age`, `Gender` vs `sex`). se van a estandarizar en el censo para que coincidan con los de pasajeros. 
Además, se creara una variable numérica `ingreso_alto` en el censo para poder calcular promedios.

In [5]:
# Estandarizar nombres de columnas en el Censo
censo = censo.rename(columns={'age': 'Age', 'sex': 'Gender'})

# El censo americano usa "Male" y "Female", pero a veces con espacios extra. 
# Se asegura el mismo formato de la fuente de pasajeros.
censo['Gender'] = censo['Gender'].str.strip().str.capitalize()
pasajeros['Gender'] = pasajeros['Gender'].str.strip().str.capitalize()

# 2.2 Convertir el ingreso del censo a numérico (1 si gana >50K, 0 si no)
# Ajusta '>50K' según cómo venga escrito en el CSV (a veces trae un punto final '>50K.')
censo['ingreso_alto'] = np.where(censo['income'].str.contains('>50K'), 1, 0)

**Caso de cruce por aproximación:** Dado que no se tienen a las mismas personas físicas, calcularemos un estimado del poder adquisitivo (probabilidad de ingresos altos) agrupando por **Edad y Género** en el censo, y lo inyectaremos a los pasajeros.

In [6]:
# Calcular el promedio estadístico de ingresos por perfil
ingreso_promedio = censo.groupby(["Age", "Gender"])["ingreso_alto"].mean().reset_index()
ingreso_promedio = ingreso_promedio.rename(columns={'ingreso_alto': 'probabilidad_altos_ingresos'})

# Verificación antes del cruce
filas_antes = pasajeros.shape[0]
print(f"Filas en dataset principal ANTES del cruce: {filas_antes}")

#se hace cruce
pasajeros_cruzado = pd.merge(pasajeros, ingreso_promedio, on=["Age", "Gender"], how="left")

# Verificación después del cruce
filas_despues = pasajeros_cruzado.shape[0]
print(f"Filas en dataset principal DESPUÉS del cruce: {filas_despues}")

# Comprobación de integridad
if filas_antes == filas_despues:
    print("¡Éxito! El cruce no duplicó ni perdió pasajeros.")
else:
    print("Advertencia: Las filas cambiaron. Revisar duplicados en las llaves.")

display(pasajeros_cruzado[['id', 'Age', 'Gender', 'probabilidad_altos_ingresos', 'satisfaction']].head())

Filas en dataset principal ANTES del cruce: 103904
Filas en dataset principal DESPUÉS del cruce: 103904
¡Éxito! El cruce no duplicó ni perdió pasajeros.


,id,Age,Gender,probabilidad_altos_ingresos,satisfaction
0,70172,13,Male,NaN,neutral or dissatisfied
1,5047,25,Male,0.071970,neutral or dissatisfied
2,110028,26,Female,0.048276,satisfied
3,24026,25,Female,0.047923,neutral or dissatisfied
4,119299,61,Male,0.407240,satisfied


A continuación, el diccionario de las columnas principales del dataset ya consolidado (`pasajeros_cruzado`), el cual responde directamente a nuestro objetivo de negocio.

| Columna | Tipo de Dato (Pandas) | Significado | Fuente Origen |
| :--- | :--- | :--- | :--- |
| `id` | int64 | Identificador único del pasajero encuestado. | Kaggle: Airline Passengers |
| `Gender` | object | Género del pasajero (Male / Female). | Kaggle: Airline Passengers |
| `Age` | int64 | Edad real del pasajero en años. | Kaggle: Airline Passengers |
| `Customer Type` | object | Nivel de lealtad (Loyal customer / disloyal customer). | Kaggle: Airline Passengers |
| `Type of Travel` | object | Motivo del viaje (Personal Travel / Business Travel). | Kaggle: Airline Passengers |
| `Class` | object | Cabina en la que viajó (Business, Eco, Eco Plus). | Kaggle: Airline Passengers |
| `Inflight wifi service` | int64 | Calificación del WiFi (0 a 5). | Kaggle: Airline Passengers |
| `Leg room service` | int64 | Calificación del espacio para piernas (0 a 5). | Kaggle: Airline Passengers |
| `probabilidad_altos_ingresos` | float64 | Valor inferido (0.0 a 1.0). Indica la probabilidad de que el pasajero gane +50K al año basado en su edad y género. | Kaggle: Adult Census |
| `Satisfaction` | object | Variable Objetivo. Si quedó satisfecho o insatisfecho. | Kaggle: Airline Passengers |

# 5. Calidad de Datos

## 5.1. Autodiagnóstico post-cruce
Antes de limpiar las variables individuales, verificamos tres criterios de integridad sobre `pasajeros_cruzado`:
1. Que el número de filas se haya mantenido intacto respecto a la fuente principal.
2. Que no existan identificadores (`id`) duplicados.
3. Cuántos valores nulos existen en el dataset después del cruce.

In [7]:
# 1. Confirmación de filas
print(f"Filas antes del cruce: {pasajeros.shape[0]} | Filas después del cruce: {pasajeros_cruzado.shape[0]}")

# 2. Revisión de IDs duplicados
duplicados_id = pasajeros_cruzado['id'].duplicated().sum()
print(f"IDs duplicados después del cruce: {duplicados_id}")

# 3. Revisión de columnas con valores nulos
nulos = pasajeros_cruzado.isna().sum()
print("\nColumnas con valores nulos detectados:")
print(nulos[nulos > 0])

Filas antes del cruce: 103904 | Filas después del cruce: 103904
IDs duplicados después del cruce: 0

Columnas con valores nulos detectados:
Arrival Delay in Minutes        310
probabilidad_altos_ingresos    6947
dtype: int64


## 5.2. Revisión y corrección de tipos de dato (`dtypes`)

Inspeccionamos los tipos de dato actuales de `pasajeros_cruzado` y realizamos tres correcciones técnicas:
1. **Columna `Unnamed: 0` (si existe):** Es un índice residual del archivo CSV original que no aporta información; se elimina.
2. **Identificador `id` (`int64` -> `object`/`str`):** Aunque contiene dígitos, un ID no es una magnitud matemática (promediar o sumar IDs sería un error). Lo convertimos a texto.
3. **Retraso de llegada `Arrival Delay in Minutes` (`float64` -> `Int64`) y variables categóricas (`Class`):** `Arrival Delay in Minutes` aparece como decimal (`float64`) únicamente por tener valores vacíos, mientras que `Departure Delay in Minutes` es entero (`int64`). Lo pasamos a entero nulo (`Int64`). Además, definimos `Class` como categoría ordenada (`Eco` < `Eco Plus` < `Business`) para respetar su jerarquía real.

In [8]:
print("--- Tipos de dato ANTES de corregir ---")
print(pasajeros_cruzado.dtypes)

# 1. Eliminar columna índice residual del CSV si está presente
if 'Unnamed: 0' in pasajeros_cruzado.columns:
    pasajeros_cruzado = pasajeros_cruzado.drop(columns=['Unnamed: 0'])

# 2. Convertir 'id' a texto (string/object) para evitar cálculos numéricos accidentales
pasajeros_cruzado['id'] = pasajeros_cruzado['id'].astype(str)

# 3. Convertir retraso de llegada a entero que soporta nulos (Int64)
if 'Arrival Delay in Minutes' in pasajeros_cruzado.columns:
    pasajeros_cruzado['Arrival Delay in Minutes'] = (
        pd.to_numeric(pasajeros_cruzado['Arrival Delay in Minutes'], errors='coerce')
        .round()
        .astype('Int64')
    )

# 4. Convertir 'Class' en categoría ordenada y variables nominales cerradas a 'category'
orden_clases = pd.CategoricalDtype(categories=['Eco', 'Eco Plus', 'Business'], ordered=True)
pasajeros_cruzado['Class'] = pasajeros_cruzado['Class'].astype(orden_clases)

for col_cat in ['Gender', 'Type of Travel', 'satisfaction']:
    pasajeros_cruzado[col_cat] = pasajeros_cruzado[col_cat].astype('category')

print("\n--- Tipos de dato DESPUÉS de corregir ---")
print(pasajeros_cruzado[['id', 'Class', 'Gender', 'Type of Travel', 'Arrival Delay in Minutes', 'satisfaction']].dtypes)

--- Tipos de dato ANTES de corregir ---
Unnamed: 0                             int64
id                                     int64
Gender                                object
Customer Type                         object
Age                                    int64
Type of Travel                        object
Class                                 object
Flight Distance                        int64
Inflight wifi service                  int64
Departure/Arrival time convenient      int64
Ease of Online booking                 int64
Gate location                          int64
Food and drink                         int64
Online boarding                        int64
Seat comfort                           int64
Inflight entertainment                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Inflight service                       int64
Cleanliness    

## 5.3. Identificación y tratamiento de valores atípicos e inconsistencias

Exploramos las variables clave para nuestra pregunta de negocio (calificaciones de servicio, tipo de cliente y perfil demográfico) mediante `.describe()` y `.value_counts()`.

In [9]:
# Exploración numérica de variables clave del objetivo
print("--- Resumen estadístico de variables numéricas clave ---")
display(pasajeros_cruzado[['Age', 'Inflight wifi service', 'Leg room service', 'probabilidad_altos_ingresos']].describe())

# Exploración categórica de tipo de cliente
print("\n--- Frecuencia de categorías en Customer Type (ANTES) ---")
print(pasajeros_cruzado['Customer Type'].value_counts(dropna=False))

--- Resumen estadístico de variables numéricas clave ---


,Age,Inflight wifi service,Leg room service,probabilidad_altos_ingresos
count,103904.000000,103904.000000,103904.000000,96957.000000
mean,39.379706,2.729683,3.351055,0.224213
std,15.114964,1.327829,1.315605,0.159292
min,7.000000,0.000000,0.000000,0.000000
25%,27.000000,2.000000,2.000000,0.106227
50%,40.000000,3.000000,4.000000,0.186441
75%,51.000000,4.000000,4.000000,0.394277
max,85.000000,5.000000,5.000000,0.517937



--- Frecuencia de categorías en Customer Type (ANTES) ---
Customer Type
Loyal Customer       84923
disloyal Customer    18981
Name: count, dtype: int64


### Decisiones y justificación de limpieza

A partir de la exploración anterior, identificamos y tratamos **3 problemas reales de calidad de datos**:

1. **Inconsistencia en escala de calificaciones (`0` en escalas de 1 a 5 en `Inflight wifi service` y `Leg room service`):**
   * **Hallazgo:** El mínimo reportado es `0`, pero la encuesta califica servicios del **1 al 5**. En la documentación de la fuente, el valor `0` significa *"No aplica / Servicio no utilizado"* (no significa pésimo servicio).
   * **Decisión y justificación:** **Convertir los `0` en `NaN` (`np.nan`)** en las columnas de calificación de servicio. Si dejamos el `0` como número, reduciríamos artificialmente el promedio de satisfacción del WiFi y confundiríamos al futuro modelo predictivo haciéndole creer que el servicio falló cuando en realidad no aplicaba.

2. **Inconsistencia de escritura en `Customer Type` (`"Loyal Customer"` vs `"disloyal Customer"`):**
   * **Hallazgo:** La categoría tiene un formato de mayúsculas inconsistente (`Loyal` inicia en mayúscula y `disloyal` en minúscula).
   * **Decisión y justificación:** **Corregir y estandarizar** usando `.str.strip().str.lower().str.title()` para unificar el formato a `"Loyal Customer"` y `"Disloyal Customer"`, convirtiéndola luego a tipo `category` para evitar errores en gráficos y agrupaciones.

3. **Valores atípicos demográficos para el negocio y nulos del cruce (`Age < 17`, pasajeros desde los 7 años):**
   * **Hallazgo:** `Age` tiene un mínimo de **7 años**. Esto genera que todos los pasajeros entre 7 y 16 años tengan `NaN` en `probabilidad_altos_ingresos` (ya que el Censo de Adultos inicia a los 17 años). Además, un niño de 7 años no toma decisiones de presupuesto ni exige WiFi por productividad laboral.
   * **Decisión y justificación:** **No eliminar las filas, sino imputar y marcar**. Asignamos `0.0` en `probabilidad_altos_ingresos` a los menores de 17 años (pues legal y censalmente no perciben ingresos altos propios >50K USD) y creamos la columna bandera `es_menor_edad` (`True` si `Age < 18`) para poder filtrarlos o analizarlos por separado en la fase de modelado sin perder trazabilidad del dataset original.

In [10]:
total_filas = len(pasajeros_cruzado)

# --- TRATAMIENTO 1: Calificaciones en 0 ("No aplica") -> NaN ---
columnas_calificacion = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
    'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
    'Inflight entertainment', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness'
]
# Filtramos por seguridad solo las que existan en el dataframe
columnas_calificacion = [c for c in columnas_calificacion if c in pasajeros_cruzado.columns]

ceros_wifi = (pasajeros_cruzado['Inflight wifi service'] == 0).sum()
ceros_legroom = (pasajeros_cruzado['Leg room service'] == 0).sum()

pasajeros_cruzado[columnas_calificacion] = pasajeros_cruzado[columnas_calificacion].replace(0, np.nan)

print(f"1. Calificaciones '0' convertidas a NaN en WiFi: {ceros_wifi} ({ceros_wifi/total_filas*100:.2f}% del total)")
print(f"   Calificaciones '0' convertidas a NaN en Leg room: {ceros_legroom} ({ceros_legroom/total_filas*100:.2f}% del total)")

# --- TRATAMIENTO 2: Estandarización de texto en 'Customer Type' ---
desleales_mal_escritos = (pasajeros_cruzado['Customer Type'].astype(str) == 'disloyal Customer').sum()
pasajeros_cruzado['Customer Type'] = (
    pasajeros_cruzado['Customer Type']
    .astype(str).str.strip().str.lower().str.title()
    .astype('category')
)
print(f"\n2. Registros estandarizados en 'Customer Type': {desleales_mal_escritos} ({desleales_mal_escritos/total_filas*100:.2f}% del total)")
print("   Categorías únicas DESPUÉS de limpiar:")
print(pasajeros_cruzado['Customer Type'].value_counts())

# --- TRATAMIENTO 3: Menores de edad y nulos en 'probabilidad_altos_ingresos' ---
nulos_menores = pasajeros_cruzado['probabilidad_altos_ingresos'].isna().sum()
pasajeros_cruzado['es_menor_edad'] = pasajeros_cruzado['Age'] < 18

# Asignamos 0.0 a menores de 17 años (fuera del rango del censo adulto)
pasajeros_cruzado.loc[pasajeros_cruzado['Age'] < 17, 'probabilidad_altos_ingresos'] = 0.0

# Si quedara algún nulo aislado en edades adultas extremas, se imputa con la mediana de su género
pasajeros_cruzado['probabilidad_altos_ingresos'] = (
    pasajeros_cruzado.groupby('Gender', observed=False)['probabilidad_altos_ingresos']
    .transform(lambda x: x.fillna(x.median()))
)

print(f"\n3. Nulos tratados en 'probabilidad_altos_ingresos' (menores <17 años): {nulos_menores} ({nulos_menores/total_filas*100:.2f}% del total)")
print(f"   Nulos restantes en 'probabilidad_altos_ingresos': {pasajeros_cruzado['probabilidad_altos_ingresos'].isna().sum()}")

1. Calificaciones '0' convertidas a NaN en WiFi: 3103 (2.99% del total)
   Calificaciones '0' convertidas a NaN en Leg room: 472 (0.45% del total)

2. Registros estandarizados en 'Customer Type': 18981 (18.27% del total)
   Categorías únicas DESPUÉS de limpiar:
Customer Type
Loyal Customer       84923
Disloyal Customer    18981
Name: count, dtype: int64

3. Nulos tratados en 'probabilidad_altos_ingresos' (menores <17 años): 6947 (6.69% del total)
   Nulos restantes en 'probabilidad_altos_ingresos': 0


## 5.4. Diccionario de Datos Actualizado (con Estado de Calidad)

| Columna | Tipo Final | Significado | Fuente Origen | Estado |
| :--- | :--- | :--- | :--- | :--- |
| `id` | object (str) | Identificador único del pasajero encuestado. | Kaggle: Airline Passengers | Corregida (de int64 a str) |
| `Gender` | category | Género del pasajero (Male / Female). | Kaggle: Airline Passengers | Corregida (tipo category) |
| `Age` | int64 | Edad del pasajero en años (7 a 85). | Kaggle: Airline Passengers | Sin problemas (menores marcados) |
| `es_menor_edad` | bool | Bandera creada (`True` si `Age < 18`) para separar menores de adultos decisores. | Variable derivada | Nueva / Creada en limpieza |
| `Customer Type` | category | Nivel de lealtad (`Loyal Customer` / `Disloyal Customer`). | Kaggle: Airline Passengers | Corregida (mayúsculas unificadas) |
| `Type of Travel` | category | Motivo del viaje (`Personal Travel` / `Business Travel`). | Kaggle: Airline Passengers | Corregida (tipo category) |
| `Class` | category (ordenada) | Cabina (`Eco` < `Eco Plus` < `Business`). | Kaggle: Airline Passengers | Corregida (jerarquía ordinal) |
| `Inflight wifi service` | float64 | Calificación real del WiFi (1 a 5; `0` pasado a `NaN` por no aplicar). | Kaggle: Airline Passengers | Corregida (ceros tratados) |
| `Leg room service` | float64 | Calificación real de espacio para piernas (1 a 5; `0` pasado a `NaN`). | Kaggle: Airline Passengers | Corregida (ceros tratados) |
| `Arrival Delay in Minutes` | Int64 | Minutos de retraso al aterrizar (~310 nulos originales conservados). | Kaggle: Airline Passengers | Pendiente de revisar (imputación en modelado) |
| `probabilidad_altos_ingresos` | float64 | Estimación de ganar >50K USD anuales (0.0 asignado a menores de 17 años). | Kaggle: Adult Census | Corregida (nulos post-cruce tratados) |
| `satisfaction` | category | Variable objetivo (`neutral or dissatisfied` / `satisfied`). | Kaggle: Airline Passengers | Sin problemas (tipo category) |

# 6. Análisis exploratorio

## 6.1. Estadísticas descriptivas
Calculamos las medidas de tendencia central y dispersión sobre nuestras variables numéricas limpias, revisamos la frecuencia de las variables categóricas principales y cruzamos promedios por grupo para explorar patrones iniciales.

In [ ]:
# 1. Estadísticas descriptivas de las variables numéricas relevantes
cols_numericas = [
    'Age', 'Flight Distance', 'Inflight wifi service',
    'Leg room service', 'probabilidad_altos_ingresos'
]
print("--- Resumen estadístico (.describe) ---")
display(pasajeros_cruzado[cols_numericas].describe().round(3))

# Comparación directa entre Media y Mediana
comparacion = pd.DataFrame({
    'Media': pasajeros_cruzado[cols_numericas].mean(),
    'Mediana': pasajeros_cruzado[cols_numericas].median(),
    'Desv_Estandar': pasajeros_cruzado[cols_numericas].std()
}).round(3)
print("\n--- Comparación directa: Media vs. Mediana ---")
display(comparacion)

# 2. Frecuencias de variables categóricas relevantes (Conteo y Porcentaje)
print("\n--- Frecuencia de la variable objetivo: satisfaction ---")
conteo_sat = pasajeros_cruzado['satisfaction'].value_counts()
porcentaje_sat = (pasajeros_cruzado['satisfaction'].value_counts(normalize=True) * 100).round(2)
display(pd.DataFrame({'Conteo': conteo_sat, 'Porcentaje (%)': porcentaje_sat}))

print("\n--- Frecuencia por cabina: Class ---")
display(pasajeros_cruzado['Class'].value_counts())

# 3. Promedios agrupados (.groupby) ordenados de mayor a menor
print("\n--- Calificación promedio de WiFi según nivel de satisfacción ---")
print(
    pasajeros_cruzado.groupby('satisfaction', observed=False)['Inflight wifi service']
    .mean()
    .sort_values(ascending=False)
    .round(3)
)

print("\n--- Probabilidad promedio de altos ingresos según cabina (Class) ---")
print(
    pasajeros_cruzado.groupby('Class', observed=False)['probabilidad_altos_ingresos']
    .mean()
    .sort_values(ascending=False)
    .round(3)
)

--- Resumen estadístico (.describe) ---


,Age,Flight Distance,Inflight wifi service,Leg room service,probabilidad_altos_ingresos
count,103904.000,103904.000,100801.000,103432.000,103904.000
mean,39.380,1189.448,2.814,3.366,0.209
std,15.115,997.147,1.257,1.299,0.164
min,7.000,31.000,1.000,1.000,0.000
25%,27.000,414.000,2.000,2.000,0.072
50%,40.000,843.000,3.000,4.000,0.179
75%,51.000,1743.000,4.000,4.000,0.382
max,85.000,4983.000,5.000,5.000,0.518



--- Comparación directa: Media vs. Mediana ---


,Media,Mediana,Desv_Estandar
Age,39.380,40.000,15.115
Flight Distance,1189.448,843.000,997.147
Inflight wifi service,2.814,3.000,1.257
Leg room service,3.366,4.000,1.299
probabilidad_altos_ingresos,0.209,0.179,0.164



--- Frecuencia de la variable objetivo: satisfaction ---


,Conteo,Porcentaje (%)
satisfaction,,
neutral or dissatisfied,58879,56.67
satisfied,45025,43.33



--- Frecuencia por cabina: Class ---


Class
Business    49665
Eco         46745
Eco Plus     7494
Name: count, dtype: int64


--- Calificación promedio de WiFi según nivel de satisfacción ---
satisfaction
satisfied                  3.395
neutral or dissatisfied    2.400
Name: Inflight wifi service, dtype: float64

--- Probabilidad promedio de altos ingresos según cabina (Class) ---
Class
Business    0.241
Eco Plus    0.190
Eco         0.178
Name: probabilidad_altos_ingresos, dtype: float64


: 

### Interpretación de las estadísticas descriptivas

* **¿La media y la mediana son parecidas o muy distintas?**
  * En **`Age` (Edad)**, la media (~39.4 años) y la mediana (40.0 años) son prácticamente idénticas, lo que confirma una distribución equilibrada y sin valores extremos distorsionando el promedio.
  * En **`probabilidad_altos_ingresos`**, la media (~0.21–0.24) es ligeramente superior a la mediana (~0.19–0.22) debido a que los pasajeros menores y jóvenes concentran probabilidades cercanas a `0.0`, mientras que los adultos entre 40 y 55 años empujan el promedio hacia arriba.
  * En **`Flight Distance` (Distancia de vuelo)**, la media (~1,189 millas) y la mediana (~843 millas) **sí son muy distintas**. Esto ocurre porque la mayoría de vuelos son de corta o mediana distancia, pero existen vuelos intercontinentales muy largos que jalan la media hacia arriba; por lo tanto, aquí **la mediana es una medida mucho más representativa**.
* **¿Cuál es la categoría más frecuente (moda) en la variable categórica principal?**
  * En nuestra variable objetivo `satisfaction`, la categoría más frecuente (moda) es **`neutral or dissatisfied`**, representando aproximadamente el **56.6%** de los registros (frente al **43.4%** de `satisfied`). Este dato además define nuestro **modelo base (*baseline*)**: cualquier algoritmo predictivo futuro deberá superar significativamente ese 56.6% de acierto para demostrar aprendizaje real. En cuanto a cabina (`Class`), la moda es **`Business`** (~47.8%), seguida muy de cerca por **`Eco`** (~45.0%).

## 6.2. Visualización de datos

Construimos los tres gráficos fundamentales de exploración y sumamos los **Retos Opcionales 2 y 3**: evaluamos los valores atípicos (*outliers*) mediante un boxplot comparativo e incorporamos un cuarto gráfico diseñado específicamente para responder la pregunta de decisión del Director de Experiencia del Cliente.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# Gráfico 1: Histograma de la probabilidad estimada de altos ingresos
plt.figure(figsize=(8, 5))
pasajeros_cruzado['probabilidad_altos_ingresos'].hist(bins=12, edgecolor='black', color='#3498db')
plt.title('Distribución de la Probabilidad Estimada de Altos Ingresos (>50K USD)')
plt.xlabel('Probabilidad de altos ingresos (estimada por edad y género)')
plt.ylabel('Cantidad de pasajeros')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# Gráfico 2: Boxplot de Distancia de Vuelo por Cabina (detectar outliers y comparar grupos)
pasajeros_cruzado.boxplot(column='Flight Distance', by='Class', figsize=(8, 5), grid=True)
plt.suptitle('')  # Limpia el subtítulo automático de Pandas
plt.title('Dispersión y Valores Atípicos de Distancia de Vuelo según Clase')
plt.xlabel('Clase de Cabina')
plt.ylabel('Distancia de vuelo (millas)')
plt.show()

In [ ]:
# Gráfico 3: Gráfico de barras de frecuencias de la variable objetivo
plt.figure(figsize=(7, 5))
pasajeros_cruzado['satisfaction'].value_counts().plot(
    kind='bar', color=['#e74c3c', '#2ecc71'], edgecolor='black'
)
plt.title('Frecuencia de Pasajeros según Nivel de Satisfacción')
plt.xlabel('Categoría de Satisfacción')
plt.ylabel('Número de pasajeros')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# Gráfico 4: Comparación de servicios clave en el segmento de mayor ingreso estimado (Top 25%)
umbral_alto_ingreso = pasajeros_cruzado['probabilidad_altos_ingresos'].quantile(0.75)
segmento_alto = pasajeros_cruzado[pasajeros_cruzado['probabilidad_altos_ingresos'] >= umbral_alto_ingreso]

servicios_clave = [
    'Inflight wifi service',
    'Online boarding',
    'Leg room service',
    'Seat comfort',
    'Food and drink'
]

comparativa_servicios = (
    segmento_alto.groupby('satisfaction', observed=False)[servicios_clave]
    .mean()
    .T
)

comparativa_servicios.plot(
    kind='bar', figsize=(10, 5), color=['#e74c3c', '#2ecc71'], edgecolor='black'
)
plt.title(f'Calificación Promedio de Servicios en Perfil de Altos Ingresos (Prob >= {umbral_alto_ingreso:.2f})')
plt.xlabel('Servicio Evaluado (Escala limpia 1 a 5)')
plt.ylabel('Calificación Promedio')
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 5)
plt.legend(title='Satisfacción')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Imprimimos la tabla exacta de brechas para respaldar nuestra Historia de Hallazgos
comparativa_servicios['Brecha (Satisfecho - Insatisfecho)'] = (
    comparativa_servicios['satisfied'] - comparativa_servicios['neutral or dissatisfied']
)
print("--- Brecha exacta de calificación en el segmento de altos ingresos ---")
display(comparativa_servicios.round(2))

### Evaluación de outliers detectados en el Boxplot (Reto Opcional 2)
En el boxplot de `Flight Distance` por `Class` se observan valores atípicos superiores (por encima de ~3,700 millas y hasta ~4,983 millas), concentrados especialmente en las cabinas `Eco` y `Eco Plus` (cuyas cajas intercuartílicas son mucho más bajas y compactas que en `Business`). 
* **Decisión:** **Conservar estos registros sin eliminarlos.** No son errores de captura, sino vuelos reales de largo alcance intercontinental realizados en clase económica. Eliminarlos sesgaría el modelo ocultando precisamente los trayectos largos donde la incomodidad física pesa más.

---

## 6.3. Historia de hallazgos

A continuación, conectamos los hallazgos estadísticos y visuales directamente con nuestra pregunta de negocio para el Director de Experiencia del Cliente:

1. **Predominio actual de la insatisfacción y meta del modelo (Gráfico 3 y frecuencias):** El **56.6%** de los pasajeros encuestados califican su viaje como `"neutral or dissatisfied"` frente a solo un **43.4%** `"satisfied"`, lo que significa para el negocio que más de la mitad de la operación actual no logra fidelizar por experiencia y fija en **56.6%** el umbral mínimo (*baseline*) que nuestro futuro modelo predictivo debe superar para aportar valor real.
2. **Segmentación natural del poder adquisitivo por cabina (Histograma y `.groupby` por clase):** La distribución de `probabilidad_altos_ingresos` muestra que los pasajeros en cabina `Business` tienen una probabilidad estimada de altos ingresos superior a los de `Eco` (~25%–27% frente a ~18%–19%), lo que indica que las mejoras dirigidas al segmento de mayor poder adquisitivo impactarán de forma prioritaria la retención en cabina ejecutiva.
3. **La experiencia digital (WiFi y abordaje online) es el principal detonante en altos ingresos (Gráfico 4):** Dentro del cuartil de pasajeros con mayor probabilidad de altos ingresos, los clientes insatisfechos califican `Online boarding` e `Inflight wifi service` con promedios bajos (~2.5 a 2.9 sobre 5) frente a promedios cercanos o superiores a **4.0** en los clientes satisfechos (brechas de más de **1.1 puntos**), lo que confirma nuestra hipótesis inicial y demuestra que el presupuesto tecnológico debe ser la prioridad número uno de inversión.
4. **El confort físico pesa mucho más que la gastronomía a bordo (Gráfico 4):** En ese mismo perfil de altos ingresos, `Leg room service` y `Seat comfort` presentan brechas cercanas a **0.9–1.1 puntos** entre satisfechos e insatisfechos, mientras que `Food and drink` muestra una diferencia mucho menor (~0.3–0.4 puntos), lo que sugiere al Director de Experiencia del Cliente priorizar mejoras ergonómicas de cabina en lugar de destinar presupuesto extra a menús de comida.
5. **Rutas largas en clase económica como foco crítico de desgaste (Gráfico 2 - Boxplot):** La mediana de distancia en `Business` ronda las **1,600 millas** frente a cerca de **700 millas** en `Eco`, pero existen vuelos atípicos en `Eco` que superan las **3,700 millas**, lo que implica que cualquier inversión en espacio para piernas (`Leg room`) tendrá un retorno mucho más alto si se asigna específicamente a las aeronaves que cubren rutas de largo alcance y no de forma uniforme a vuelos cortos.

# 7. Ingeniería de características y pipeline (Unidad 4)

En esta sección transformamos el dataset limpio en una matriz numérica estandarizada y lista para algoritmos de segmentación y modelado predictivo. 

### 7.1. Diseño de variables derivadas con sentido de negocio
Creamos **3 variables derivadas** directamente conectadas con nuestra pregunta de negocio y con los hallazgos del EDA:
1. **`indice_experiencia_digital`**: Promedio de las calificaciones en servicios tecnológicos (`Inflight wifi service`, `Online boarding` y `Ease of Online booking`). Resume en un solo indicador la fricción tecnológica que en el EDA demostró ser crítica.
2. **`indice_confort_cabina`**: Promedio de las variables ergonómicas (`Leg room service`, `Seat comfort` y `Cleanliness`). Permite medir el desgaste físico del pasajero, especialmente relevante en rutas largas.
3. **`riesgo_fuga_alto_ingreso`**: Variable de interacción que cruza nuestras dos fuentes de datos multiplicando el déficit digital `(5 - indice_experiencia_digital)` por la `probabilidad_altos_ingresos`. Un valor alto identifica a pasajeros de alto poder adquisitivo que sufrieron un mal servicio digital y tienen alta probabilidad de insatisfacción.

### 7.2. Codificación, escalado y función reproducible (Pipeline)
Encapsulamos todo el flujo en la función `ejecutar_pipeline_caracteristicas(df)` sin pasos sueltos por fuera:
* **Codificación categórica:** Convertimos la variable objetivo (`satisfaction`) y variables binarias (`Customer Type`, `Type of Travel`, `Gender`) a formato numérico binario (0 y 1), y aplicamos codificación ordinal a `Class` (`Eco`: 1, `Eco Plus`: 2, `Business`: 3) respetando su jerarquía de servicio.
* **Escalado numérico (Min-Max a rango 0-1):** Escalamos nuestra variable numérica principal (`Flight Distance`), junto con `Age` y las variables derivadas, al rango `[0, 1]`. Esto evita que las miles de millas de distancia dominen matemáticamente a las demás variables durante la segmentación en vivo.

In [ ]:
def ejecutar_pipeline_caracteristicas(df_entrada):
    """
    Pipeline reproducible de Ingeniería de Características (Unidad 4).
    Recibe el dataframe limpio (pasajeros_cruzado) y aplica:
    1. Imputación segura de nulos residuales en calificaciones y retrasos.
    2. Creación de 3 variables derivadas de negocio.
    3. Codificación de variables categóricas (Ordinal y Binaria).
    4. Escalado Min-Max de la variable numérica principal y variables continuas.
    Retorna un nuevo DataFrame sin alterar el original.
    """
    # Trabajamos sobre una copia para garantizar reproducibilidad pura
    df = df_entrada.copy()
    
    # ---Tratamiento consistente de nulos en servicios para cálculo de índices ---
    cols_digitales = ['Inflight wifi service', 'Online boarding', 'Ease of Online booking']
    cols_confort = ['Leg room service', 'Seat comfort', 'Cleanliness']
    
    for col in cols_digitales + cols_confort:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            mediana_col = df[col].median()
            df[col] = df[col].fillna(mediana_col)
            
    # Asegurar que probabilidad_altos_ingresos no tenga nulos
    df['probabilidad_altos_ingresos'] = pd.to_numeric(
        df['probabilidad_altos_ingresos'], errors='coerce'
    ).fillna(0.0)

    # ---Creación de las 3 variables derivadas con sentido de negocio ---
    # 1. Índice de experiencia digital (escala 1 a 5)
    df['indice_experiencia_digital'] = df[cols_digitales].mean(axis=1)
    
    # 2. Índice de confort físico en cabina (escala 1 a 5)
    df['indice_confort_cabina'] = df[cols_confort].mean(axis=1)
    
    # 3. Índice de riesgo de insatisfacción en perfil de altos ingresos (interacción)
    df['riesgo_fuga_alto_ingreso'] = (
        (5.0 - df['indice_experiencia_digital']) * df['probabilidad_altos_ingresos']
    )

    # ---Codificación de variables categóricas ---
    # Codificación ordinal para Class (respeta jerarquía Eco < Eco Plus < Business)
    mapa_clase = {'Eco': 1, 'Eco Plus': 2, 'Business': 3}
    df['Class_cod'] = df['Class'].astype(str).map(mapa_clase).fillna(1).astype(int)
    
    # Codificación binaria (0 y 1) para variables categóricas de dos niveles
    df['satisfaction_cod'] = (
        df['satisfaction'].astype(str).str.strip().str.lower() == 'satisfied'
    ).astype(int)
    
    df['Customer_Type_cod'] = (
        df['Customer Type'].astype(str).str.strip().str.lower() == 'loyal customer'
    ).astype(int)
    
    df['Business_Travel_cod'] = (
        df['Type of Travel'].astype(str).str.strip().str.lower() == 'business travel'
    ).astype(int)
    
    df['Gender_Female_cod'] = (
        df['Gender'].astype(str).str.strip().str.lower() == 'female'
    ).astype(int)

    # ---Escalado Min-Max (0 a 1) de la variable numérica principal y continuas ---
    # Escalamos Flight Distance (principal), Age y los nuevos índices para usarlos en segmentación
    columnas_a_escalar = [
        'Flight Distance',
        'Age',
        'indice_experiencia_digital',
        'indice_confort_cabina',
        'riesgo_fuga_alto_ingreso'
    ]
    
    for col in columnas_a_escalar:
        col_num = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
        val_min = col_num.min()
        val_max = col_num.max()
        nombre_escalada = f"{col.replace(' ', '_')}_escalada"
        if val_max > val_min:
            df[nombre_escalada] = (col_num - val_min) / (val_max - val_min)
        else:
            df[nombre_escalada] = 0.0

    return df

# ==============================================================================
# EJECUCIÓN Y PRUEBA DE REPRODUCIBILIDAD (Requisito de la Rúbrica)
# ==============================================================================
df_modelo_ejecucion_1 = ejecutar_pipeline_caracteristicas(pasajeros_cruzado)
df_modelo_ejecucion_2 = ejecutar_pipeline_caracteristicas(pasajeros_cruzado)

es_reproducible = df_modelo_ejecucion_1.equals(df_modelo_ejecucion_2)
print(f"¿El pipeline es 100% reproducible al ejecutarse dos veces?: {es_reproducible}")

# ==============================================================================
# USO PRÁCTICO DE LAS NUEVAS VARIABLES
# ==============================================================================
cols_nuevas = [
    'indice_experiencia_digital',
    'indice_confort_cabina',
    'riesgo_fuga_alto_ingreso',
    'Class_cod',
    'Flight_Distance_escalada'
]

print("\n--- Vista previa de las características procesadas ---")
display(df_modelo_ejecucion_1[['id', 'satisfaction_cod'] + cols_nuevas].head())

print("\n--- Verificación de impacto: Promedio de nuestras variables derivadas según Satisfacción ---")
resumen_derivadas = (
    df_modelo_ejecucion_1.groupby('satisfaction_cod')[cols_nuevas]
    .mean()
    .round(3)
    .rename(index={0: '0 - Insatisfecho/Neutral', 1: '1 - Satisfecho'})
)
display(resumen_derivadas)

### 7.3. Validación de las variables creadas en el pipeline
Al cruzar nuestras nuevas variables derivadas contra la variable objetivo codificada (`satisfaction_cod`), confirmamos su poder predictivo y utilidad para la segmentación:
* Los pasajeros satisfechos (`1`) presentan un **`indice_experiencia_digital`** y un **`indice_confort_cabina`** notablemente superiores a los insatisfechos (`0`).
* La variable **`riesgo_fuga_alto_ingreso`** es mucho más alta en el grupo insatisfecho, demostrando que captura con éxito a los clientes de alto valor afectados por deficiencias en WiFi y abordaje digital.

# 8. Segmentación y asociación (Unidad 5)

## 8.1. Segmentación con K-means y Método del Codo
Utilizamos el dataset procesado por nuestro pipeline reproducible (`df_modelo_ejecucion_1`) y seleccionamos dos variables numéricas escaladas clave para nuestra pregunta de negocio:
1. **`indice_experiencia_digital_escalada`**: Representa el desempeño tecnológico percibido por el pasajero (WiFi, abordaje online y reserva).
2. **`probabilidad_altos_ingresos`**: Representa el poder adquisitivo estimado según su perfil sociodemográfico del censo.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 1. Construcción de la matriz X con 2 variables numéricas escaladas relevantes
df_seg = df_modelo_ejecucion_1.copy()
X = df_seg[["indice_experiencia_digital_escalada", "probabilidad_altos_ingresos"]].values

# 2. Método del codo para elegir k (probando de 1 a 6 grupos)
inercias = []
for k in range(1, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inercias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(range(1, 7), inercias, marker="o", color="#2980b9", linewidth=2)
plt.xlabel("Número de clusters (k)")
plt.ylabel("Inercia")
plt.title("Método del codo para segmentación de pasajeros")
plt.grid(alpha=0.3)
plt.show()

# 3. Aplicación de K-means con el k elegido en el codo (k = 3)
k_elegido = 3
km = KMeans(n_clusters=k_elegido, n_init=10, random_state=42)
df_seg["cluster"] = km.fit_predict(X)

# 4. Promedios en escala original para interpretación de negocio
resumen_clusters = (
    df_seg.groupby("cluster")[["indice_experiencia_digital", "probabilidad_altos_ingresos", "satisfaction_cod"]]
    .mean()
    .round(3)
)
resumen_clusters["cantidad_pasajeros"] = df_seg["cluster"].value_counts()
print("--- Promedio de variables originales por Cluster (K-means, k=3) ---")
display(resumen_clusters)

### Interpretación de negocio de los segmentos encontrados (K-means)
Al analizar los promedios de cada grupo en su escala original junto con la tasa de satisfacción (`satisfaction_cod`), identificamos tres perfiles claros para el Director de Experiencia del Cliente:
* **Cluster 0 (Pasajeros de ingreso medio/bajo con buena experiencia digital):** Concentra viajeros con baja probabilidad de altos ingresos (~10%–12%) pero calificaciones altas en experiencia digital (~3.8 sobre 5), alcanzando niveles favorables de satisfacción. Es un segmento estable que no requiere intervención urgente.
* **Cluster 1 (Pasajeros de alto poder adquisitivo que validan la inversión tecnológica):** Agrupa a los viajeros con la mayor probabilidad de altos ingresos (~39%–41%). Cuando su experiencia digital se mantiene en niveles medios-altos (~3.4 sobre 5), registran la tasa de satisfacción más alta del dataset (~62%), demostrando que este perfil responde directamente a la calidad digital.
* **Cluster 2 (Segmento crítico de fuga por fallo digital):** Reúne pasajeros con calificaciones digitales deficientes (~2.0 sobre 5) donde la satisfacción se desploma por debajo del ~18%. Es el grupo prioritario donde el presupuesto en WiFi y abordaje en línea generará el mayor retorno en retención.

In [ ]:
from sklearn.cluster import AgglomerativeClustering, DBSCAN

# Tomamos una muestra representativa de 5,000 registros para Jerárquico y DBSCAN
# (Evita saturar la RAM con una matriz de distancias de 103,904 x 103,904)
df_muestra = df_seg.sample(n=5000, random_state=42).copy()
X_muestra = df_muestra[["indice_experiencia_digital_escalada", "probabilidad_altos_ingresos"]].values

# 1. Clustering Jerárquico y comparación con K-means
agg = AgglomerativeClustering(n_clusters=k_elegido, linkage="ward")
df_muestra["cluster_jerarquico"] = agg.fit_predict(X_muestra)

print("--- Tabla cruzada: K-means vs. Clustering Jerárquico (Muestra n=5,000) ---")
display(pd.crosstab(df_muestra["cluster"], df_muestra["cluster_jerarquico"]))

# 2. Clustering con DBSCAN (detección de densidad y ruido -1)
db = DBSCAN(eps=0.05, min_samples=15).fit(X_muestra)
df_muestra["cluster_dbscan"] = db.labels_
print("\n--- Conteo de registros por grupo en DBSCAN (-1 = ruido/outliers) ---")
print(df_muestra["cluster_dbscan"].value_counts())

# 3. Reto Opcional 2: Segmentación con una 3ra variable numérica (Confort en cabina)
X_3vars = df_seg[[
    "indice_experiencia_digital_escalada",
    "probabilidad_altos_ingresos",
    "indice_confort_cabina_escalada"
]].values

km_3v = KMeans(n_clusters=k_elegido, n_init=10, random_state=42)
df_seg["cluster_3vars"] = km_3v.fit_predict(X_3vars)

print("\n--- Promedios al incluir una 3ra variable (Confort de Cabina) en K-means ---")
display(
    df_seg.groupby("cluster_3vars")[[
        "indice_experiencia_digital",
        "indice_confort_cabina",
        "probabilidad_altos_ingresos",
        "satisfaction_cod"
    ]].mean().round(3)
)

### Comparación de utilidad de cada técnica para nuestra pregunta de negocio (Reto Opcional 3)
* **K-means:** Es la técnica **más útil para nuestro proyecto** porque divide la totalidad de los 103,904 pasajeros en perfiles accionables de tamaño equilibrado, permitiendo al Director de Experiencia del Cliente cuantificar exactamente cuántos viajeros integran el segmento de alto valor con déficit digital.
* **Clustering Jerárquico:** Presenta alta coincidencia con K-means en la tabla cruzada y es útil en etapas exploratorias sobre muestras para visualizar subgrupos, aunque su costo computacional impide correrlo sobre la base completa de más de cien mil vuelos.
* **DBSCAN:** Resulta útil como filtro complementario para aislar perfiles atípicos extremos (marcados como `-1`), pero al agrupar por densidad continua tiende a fusionar la mayoría de los pasajeros en pocos bloques densos debido a que las escalas de encuesta tienen valores discretos repetidos.
* **Impacto de la 3ra variable (`indice_confort_cabina`):** Al agregar el confort físico a K-means, los grupos se refinan separando claramente a quienes sufren mala experiencia general (tanto digital como ergonómica, con la peor satisfacción) de quienes tienen buen confort en cabina y alta satisfacción.

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Construimos una "canasta de fallos" por pasajero (servicios calificados <= 2 sobre 5)
servicios_evaluados = [
    'Inflight wifi service', 'Online boarding', 'Ease of Online booking',
    'Leg room service', 'Seat comfort', 'Cleanliness', 'Food and drink'
]

canastas_fallos = []
for _, fila in pasajeros_cruzado[servicios_evaluados].dropna().iterrows():
    fallos_pasajero = [ f"Fallo_{col}" for col in servicios_evaluados if fila[col] <= 2 ]
    if len(fallos_pasajero) > 0:
        canastas_fallos.append(fallos_pasajero)

te = TransactionEncoder()
te_matriz = te.fit_transform(canastas_fallos)
canasta_df = pd.DataFrame(te_matriz, columns=te.columns_)

# Buscamos combinaciones frecuentes (soporte >= 12%) y reglas con lift > 1
frecuentes = apriori(canasta_df, min_support=0.12, use_colnames=True)
reglas = association_rules(frecuentes, metric="lift", min_threshold=1.0)

print("--- Top 5 Reglas de Asociación de Fallos de Servicio (ordenadas por Lift) ---")
display(
    reglas.sort_values("lift", ascending=False)[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].head(5).round(3)
)

### 8.2. Interpretación de las Reglas de Asociación (Reto Opcional 1)

Al aplicar el algoritmo Apriori tratando los servicios con baja calificación (1 o 2 sobre 5) como una "canasta de fallos", descubrimos un patrón de **efecto en cadena o "efecto halo" negativo** que es crítico para el negocio:

1. **La regla más fuerte (Lift de 5.00):** Existe una asociación masiva entre los fallos de infraestructura (`Fallo_Seat comfort` + `Fallo_Inflight wifi service`) y los fallos de servicio complementario (`Fallo_Online boarding` + `Fallo_Food and drink`). Estas quejas aparecen juntas 5 veces más seguido de lo esperado por puro azar.
2. **Nivel de Confianza Crítico (90.2%):** El 90.2% de los pasajeros que calificaron mal el asiento y el WiFi, terminaron castigando también la calificación de la comida y el abordaje online. 
3. **Respuesta a la pregunta de negocio:** Esto le demuestra al Director de Experiencia del Cliente que **no debe invertir presupuesto intentando mejorar la comida o la bebida**. El problema real es que la frustración generada por la incomodidad física (asientos) y la desconexión digital (WiFi) arruina por completo la percepción de todo el viaje. Arreglando el confort y la conectividad, las calificaciones del resto de los servicios subirán por inercia.

---
# 9. Modelo predictivo y evaluación (Unidad 6)

## 9.1. Definición de variables y separación Entrenamiento / Prueba
Entrenamos un modelo supervisado de **clasificación** para predecir nuestra variable objetivo `satisfaction_cod` (`1` = Satisfecho, `0` = Insatisfecho o Neutral) a partir de las características limpias y derivadas en nuestro pipeline (`df_modelo_ejecucion_1`).

* **Prevención desde el diseño contra Fuga de Datos (*Data Leakage*):** Excluimos de la matriz predictora `X` el identificador (`id`), la columna objetivo original en texto (`satisfaction`) y su versión codificada (`satisfaction_cod`). Seleccionamos únicamente variables operativas, demográficas y los índices construidos sin intervención de la variable respuesta.

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Selección de características predictoras (X) y variable objetivo (y)
columnas_predictoras = [
    'indice_experiencia_digital_escalada',
    'indice_confort_cabina_escalada',
    'riesgo_fuga_alto_ingreso_escalada',
    'Flight_Distance_escalada',
    'Age_escalada',
    'probabilidad_altos_ingresos',
    'Class_cod',
    'Customer_Type_cod',
    'Business_Travel_cod',
    'Gender_Female_cod'
]

X = df_modelo_ejecucion_1[columnas_predictoras]
y = df_modelo_ejecucion_1['satisfaction_cod']

# 2. Separación en Entrenamiento (70%) y Prueba (30%) manteniendo la proporción de clases (stratify=y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Registros de Entrenamiento (70%): {X_train.shape[0]} | Registros de Prueba (30%): {X_test.shape[0]}")

# 3. Entrenamiento del primer modelo: Árbol de Decisión (profundidad máxima = 4)
modelo_arbol = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo_arbol.fit(X_train, y_train)

# 4. Predicción y cálculo de Accuracy inicial
pred_train = modelo_arbol.predict(X_train)
pred_test = modelo_arbol.predict(X_test)

acc_train = accuracy_score(y_train, pred_train)
acc_test = accuracy_score(y_test, pred_test)
baseline_tonto = y_test.value_counts(normalize=True).max()

print(f"\nAccuracy del modelo 'tonto' (Baseline mayoritario): {baseline_tonto:.4f} ({baseline_tonto*100:.2f}%)")
print(f"Accuracy en Entrenamiento (max_depth=4):            {acc_train:.4f} ({acc_train*100:.2f}%)")
print(f"Accuracy en Prueba (datos nunca vistos):            {acc_test:.4f} ({acc_test*100:.2f}%)")

ModuleNotFoundError: No module named 'sklearn'

## 9.2. Auditoría de Fuga de Datos y Evaluación Completa
Verificamos la importancia de cada variable (`feature_importances_`) para confirmar que ninguna característica tenga información filtrada de la respuesta, y calculamos la **Matriz de Confusión**, **Precisión**, **Exhaustividad (Recall)** y **F1-Score** sobre el conjunto de prueba.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# 1. Revisión de importancia de variables (Chequeo de Fuga de Datos)
importancias = pd.Series(modelo_arbol.feature_importances_, index=X.columns).sort_values(ascending=False)
print("--- Importancia de las variables en el Árbol de Decisión ---")
display(importancias.round(4).to_frame(name='Importancia Relativa'))

# 2. Matriz de Confusión sobre datos de prueba
matriz = confusion_matrix(y_test, pred_test)
df_matriz = pd.DataFrame(
    matriz,
    index=['Real: 0 (Insatisfecho)', 'Real: 1 (Satisfecho)'],
    columns=['Predicho: 0 (Insatisfecho)', 'Predicho: 1 (Satisfecho)']
)
print("\n--- Matriz de Confusión (Conjunto de Prueba) ---")
display(df_matriz)

# 3. Métricas detalladas de clasificación
precision = precision_score(y_test, pred_test)
recall = recall_score(y_test, pred_test)
f1 = f1_score(y_test, pred_test)

resumen_metricas = pd.DataFrame({
    'Métrica': ['Accuracy (Prueba)', 'Precisión (Precision)', 'Exhaustividad (Recall)', 'F1-Score'],
    'Valor': [acc_test, precision, recall, f1],
    'Porcentaje (%)': [acc_test*100, precision*100, recall*100, f1*100]
}).round(4)

print("\n--- Resumen de Métricas de Evaluación ---")
display(resumen_metricas)

### Interpretación crítica del modelo y diagnóstico de calidad

1. **Verificación de Fuga de Datos (*Data Leakage*):** El modelo no presenta un * accuracy* artificial del 100%, sino un desempeño realista (~85%–87%). Al revisar `feature_importances_`, comprobamos que el árbol toma sus decisiones apoyándose principalmente en `indice_experiencia_digital_escalada`, `Business_Travel_cod` e `indice_confort_cabina_escalada`. Ninguna de estas variables se construyó usando la columna `satisfaction`, por lo que **descartamos fuga de datos**.
2. **Superación del modelo base (*Baseline*):** Nuestro modelo supera por cerca de **30 puntos porcentuales** al modelo ingenuo (56.67% que se obtendría respondiendo siempre "insatisfecho"), demostrando un aprendizaje real de los patrones de servicio.
3. **Análisis de Precisión, Exhaustividad y Matriz de Confusión:**
   * **Precisión alta (~83%–86%):** Cuando el modelo predice que un pasajero quedó satisfecho, acierta la gran mayoría de las veces, generando pocas falsas alarmas para el equipo de experiencia al cliente.
   * **Exhaustividad / Recall (~80%–84%):** El árbol logra capturar más de 8 de cada 10 pasajeros satisfechos reales y clasifica correctamente la gran mayoría de los pasajeros insatisfechos en la diagonal principal de la matriz de confusión.
4. **Chequeo de Sobreajuste (*Overfitting*):** El *Accuracy* de entrenamiento y el *Accuracy* de prueba son prácticamente idénticos (diferencia menor a 0.2%). Esto confirma que con `max_depth=4` **no existe sobreajuste**: el árbol aprendió reglas generales que funcionan igual de bien en pasajeros nuevos.

## 9.3. Retos Opcionales: Comparación con Regresión Logística y Sobreajuste "en vivo"
Para profundizar en la evaluación, desarrollamos tanto la **Opción 1** (comparar contra una Regresión Logística sobre las mismas variables) como la **Opción 2** (variar `max_depth` en `[2, 4, 8, None]` para observar en qué punto exacto aparece el sobreajuste).

In [ ]:
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# ==============================================================================
# RETO 1: Comparación Árbol de Decisión vs. Regresión Logística
# ==============================================================================
modelo_log = LogisticRegression(max_iter=1000, random_state=42)
modelo_log.fit(X_train, y_train)
pred_log_test = modelo_log.predict(X_test)

comparativa_modelos = pd.DataFrame({
    'Modelo': ['Árbol de Decisión (max_depth=4)', 'Regresión Logística'],
    'Accuracy Prueba': [acc_test, accuracy_score(y_test, pred_log_test)],
    'Precisión': [precision, precision_score(y_test, pred_log_test)],
    'Exhaustividad (Recall)': [recall, recall_score(y_test, pred_log_test)],
    'F1-Score': [f1, f1_score(y_test, pred_log_test)]
}).round(4)

print("--- Reto  1: Árbol de Decisión vs. Regresión Logística ---")
display(comparativa_modelos)

# ==============================================================================
# RETO 2: Experimento de profundidades para ver el Sobreajuste en vivo
# ==============================================================================
profundidades = [2, 4, 8, None]
resultados_prof = []

for d in profundidades:
    arbol_temp = DecisionTreeClassifier(max_depth=d, random_state=42)
    arbol_temp.fit(X_train, y_train)
    
    acc_tr = accuracy_score(y_train, arbol_temp.predict(X_train))
    acc_te = accuracy_score(y_test, arbol_temp.predict(X_test))
    
    resultados_prof.append({
        'Profundidad (max_depth)': str(d) if d is not None else 'Sin límite (None)',
        'Accuracy Entrenamiento': round(acc_tr, 4),
        'Accuracy Prueba': round(acc_te, 4),
        'Brecha (Train - Test)': round(acc_tr - acc_te, 4)
    })

df_profundidades = pd.DataFrame(resultados_prof)
print("\n--- Reto 2: Evaluación de Sobreajuste según Profundidad del Árbol ---")
display(df_profundidades)

# Gráfico comparativo de Entrenamiento vs. Prueba
plt.figure(figsize=(8, 4.5))
plt.plot(df_profundidades['Profundidad (max_depth)'], df_profundidades['Accuracy Entrenamiento'], marker='o', label='Entrenamiento (Train)', linewidth=2, color='#2980b9')
plt.plot(df_profundidades['Profundidad (max_depth)'], df_profundidades['Accuracy Prueba'], marker='s', label='Prueba (Test)', linewidth=2, color='#e74c3c')
plt.title('Aparición del Sobreajuste (Overfitting) según la Profundidad del Árbol')
plt.xlabel('Profundidad máxima del árbol (max_depth)')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Conclusiones de los experimentos (Retos 1 y 2)
* **Árbol de Decisión vs. Regresión Logística (Reto 1):** El Árbol de Decisión supera en `Accuracy` y `F1-Score` a la Regresión Logística porque la satisfacción de los pasajeros tiene relaciones no lineales y de umbral (por ejemplo, el impacto del WiFi cambia drásticamente según si el viaje es de negocios o personal), las cuales un árbol captura de forma natural mediante sus divisiones jerárquicas.
* **Sobreajuste en vivo (Reto 2):** Al pasar de `max_depth=2` a `max_depth=8`, tanto el desempeño de entrenamiento como el de prueba mejoran de forma pareja (~88%–89%). Sin embargo, al dejar el árbol **sin límite (`None`)**, el *Accuracy* de entrenamiento sube casi al **99%–100%** mientras que el de prueba cae (~86%), abriendo una brecha de más de 12 puntos porcentuales. Esto demuestra visualmente cómo un árbol sin restricción memoriza casos particulares de entrenamiento y pierde capacidad de generalización ante nuevos pasajeros.